# OFBO ontology and RDF-data validation

This notebook documents ontology-level and instance-data validation of the Organic Flow Battery Ontology (OFBO) and the RDF graph materialized from the Ontop OBDA mappings.

The validation workflow consists of three complementary steps:

1. **OWL reasoning with HermiT**  
   The OFBO ontology is classified using the HermiT OWL reasoner through ROBOT. This evaluates the logical consequences of the asserted ontology axioms and produces a reasoned ontology serialisation.

2. **Ontology-quality assessment with ROBOT**  
   ROBOT generates a machine-readable ontology-quality report. The report is reviewed to distinguish OFBO-specific findings from expected findings caused by imported or explicitly reused external ontology terms.

3. **SHACL validation of materialized RDF**  
   RDF materialized from the Ontop OBDA mappings is validated against OFBO SHACL constraint shapes using pySHACL version 0.40.1. The SHACL constraints assess molecular representation, quantitative values and units, redox-potential context, generating activities, provenance, pH records, and diffusion-coefficient records.

Together, these checks distinguish ontology-level logical validation from constraint-based validation of the generated RDF instances. The reports produced by this notebook are retained as reproducibility artefacts in `validation/reports/`.

## ROBOT ontology validation

Ontology-level reasoning and ontology-quality assessment were performed using ROBOT.

The ontology was classified with the HermiT OWL reasoner:

```bash
robot reason \
  --reasoner HermiT \
  --input ontology/OFBO.ttl \
  --output validation/reports/OFBO-reasoned.ttl
```

The command completed successfully and produced the reasoned ontology serialisation:
```
validation/reports/OFBO-reasoned.ttl
```

A ROBOT ontology-quality report was then generated using:

```bash
robot report \
  --input ontology/OFBO.ttl \
  --output validation/reports/robot-report.tsv \
  --fail-on none
```

The resulting machine-readable report is stored at:
```
validation/reports/robot-report.tsv
```

## ROBOT report interpretation and revision summary

The ROBOT report was reviewed manually. Its findings predominantly concern imported or explicitly reused external ontology terms and expected consequences of OFBO alignment axioms, rather than unresolved logical inconsistencies in OFBO.

### Reused and aligned terms

The report identifies duplicate labels and equivalent-class pairs for OFBO concepts explicitly aligned with external ontologies. Examples include the alignment of OFBO terms with BFO, ChEBI, COB, and PMD concepts, such as `process`, `role`, `property`, and `thermodynamic property`. Duplicate labels are expected in these cases because semantically equivalent concepts retain the same human-readable label. These alignments are intentional and support ontology reuse and interoperability.

### Definitions

ROBOT's default `missing_definition` rule checks specifically for the OBO annotation property `IAO:0000115`. OFBO definitions are represented using `skos:definition` rather than `IAO:0000115`. Therefore, the reported `missing_definition` warnings for OFBO terms do not indicate that textual definitions are absent; they reflect the annotation-property convention used by the default ROBOT profile. `skos:definition` was retained because it is an appropriate standard property for textual concept definitions and addresses the ontology-documentation requirements of the revision. Following check confirms all OFBO terms are annotates with `skos:definition`:

In [2]:
from pathlib import Path

import pandas as pd
from rdflib import Graph, Namespace
from rdflib.namespace import OWL, RDF, RDFS, SKOS


ONTOLOGY_FILE = Path("../ontology/OFBO.ttl")

OFBO = Namespace("https://w3id.org/ofbo#")

ontology = Graph()
ontology.parse(ONTOLOGY_FILE, format="turtle")

# Term types to evaluate.
TERM_TYPES = {
    OWL.Class: "Class",
    OWL.ObjectProperty: "Object property",
    OWL.DatatypeProperty: "Datatype property",
    OWL.AnnotationProperty: "Annotation property",
}

rows = []

for rdf_type, term_type in TERM_TYPES.items():
    for term in ontology.subjects(RDF.type, rdf_type):
        # Check only terms defined in the OFBO namespace.
        if not str(term).startswith(str(OFBO)):
            continue

        labels = list(ontology.objects(term, RDFS.label))
        definitions = list(ontology.objects(term, SKOS.definition))

        rows.append(
            {
                "term": str(term),
                "termType": term_type,
                "label": str(labels[0]) if labels else None,
                "hasSkosDefinition": bool(definitions),
                "definition": str(definitions[0]) if definitions else None,
            }
        )

definition_check = pd.DataFrame(rows).sort_values(
    by=["termType", "term"]
).reset_index(drop=True)

missing_definitions = definition_check[
    ~definition_check["hasSkosDefinition"]
].copy()

print(f"OFBO terms checked: {len(definition_check)}")
print(f"Terms without skos:definition: {len(missing_definitions)}")

missing_definitions[
    ["termType", "label", "term"]
]

OFBO terms checked: 138
Terms without skos:definition: 0


,termType,label,term


### External-resource findings

Missing-label, multiple-label, annotation-whitespace, missing-subset-declaration, and many missing-definition findings concern terms from external resources, including BFO, IAO, QUDT, Schema.org, FOAF, RXNO, and RO. These annotations are maintained by the respective source ontologies and are not modified by OFBO.

### Inferred upper-level hierarchy

ROBOT reports that three high-level OFBO classes have no explicitly asserted `rdfs:subClassOf` axiom. These classes are explicitly declared equivalent to external ontology classes that are already positioned within an upper-level hierarchy. Their superclass relationships are therefore inferred by reasoning through the equivalence axioms. The absence of directly asserted superclass triples is intentional and does not indicate that these classes are disconnected from the ontology hierarchy.

# SHACL validation of materialized OFBO RDF

This notebook validates the RDF graph materialized from the Ontop OBDA mappings against the OFBO SHACL constraint shapes. SHACL validation is performed using pySHACL version 0.40.1.

## Validation steps

The SHACL shapes evaluate the following requirements:

1. **Molecular representation**  
   Each instance of `ofbo:OFBO_0000012` (molecule) must have at least one molecular representation: a SMILES string, InChIKey, or IUPAC name.

2. **Quantitative values and units**  
   Each resource with `qudt:numericValue` must have exactly one decimal numeric value and exactly one unit represented by an IRI.

3. **Redox-potential records**  
   Each redox-potential record must specify:
   - exactly one bearer, represented as an electrochemical reaction;
   - exactly one decimal value;
   - exactly one unit;
   - exactly one valid potential reference;
   - exactly one generating process.

4. **Generating processes**  
   Every resource used as the object of `prov:wasGeneratedBy` must be an IRI and must be typed as a `prov:Activity`.

5. **pH records**  
   Each pH record must have one decimal value in the supported aqueous range and use the QUDT pH unit.

6. **Diffusion coefficients**  
   Each diffusion coefficient must have a positive decimal value and an accepted diffusion-coefficient unit.

7. **Provenance**  
   Each generated result must be traceable either to a primary literature source using `prov:hadPrimarySource` or, for newly generated unpublished measurements, to a responsible `prov:Agent` associated with its generating activity through `prov:wasAssociatedWith`.

8. **Canonical units**  
   Each quantitative result is checked for consistency with the canonical unit defined for its corresponding OFBO property. The result must have exactly one unit represented by an IRI through `qudt:unit`, and this unit must match the property-specific canonical unit defined in the OFBO ontology. Values reported in other units must be converted before being exposed through the OBDA mappings. SHACL validation checks these unit assignments against the canonical-unit constraints defined in `ofbo-shapes.ttl`.

The validation report is serialised as Turtle and can be inspected to identify the focus node, violated constraint, path, and validation message for each violation.

A successful SHACL result confirms conformance of the materialized RDF graph to these declared constraints.

In [ ]:
from pathlib import Path

from rdflib import Graph, Namespace
from pyshacl import validate

# Adjust these paths if the notebook is stored elsewhere.
DATA_FILE = Path("../database/triples/OFBO-materialized.ttl")
SHAPES_FILE = Path("ofbo-shapes.ttl")
REPORT_FILE = Path("reports/shacl-report.ttl")

SH = Namespace("http://www.w3.org/ns/shacl#")


def main():
    if not DATA_FILE.exists():
        raise FileNotFoundError(f"Materialized RDF file not found: {DATA_FILE.resolve()}")

    if not SHAPES_FILE.exists():
        raise FileNotFoundError(f"SHACL shapes file not found: {SHAPES_FILE.resolve()}")

    REPORT_FILE.parent.mkdir(parents=True, exist_ok=True)

    data_graph = Graph()
    data_graph.parse(DATA_FILE, format="turtle")

    shapes_graph = Graph()
    shapes_graph.parse(SHAPES_FILE, format="turtle")

    conforms, report_graph, _ = validate(
        data_graph=data_graph,
        shacl_graph=shapes_graph,
        inference="none",
        abort_on_first=False,
        allow_infos=True,
        allow_warnings=True,
        meta_shacl=True,
        advanced=True,
    )

    report_graph.serialize(destination=REPORT_FILE, format="turtle")

    violation_count = sum(
        1 for _ in report_graph.subjects(predicate=SH.resultSeverity, object=SH.Violation)
    )

    print(f"SHACL violations: {violation_count}")
    print(f"SHACL report saved to: {REPORT_FILE.resolve()}")

    if not conforms:
        raise SystemExit("SHACL validation failed. Inspect the report for details.")


main()

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/home/dwillimetz/.local/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/home/dwillimetz/.local/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/home/dwillimet

SHACL violations: 168
SHACL report saved to: /home/dwillimetz/Desktop/OFBO/validation/reports/shacl-report.ttl


/home/dwillimetz/.local/lib/python3.13/site-packages/rdflib/term.py:1715: UserWarning: Serializing weird numerical rdflib.term.Literal('< 0.01', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal'))
  warnings.warn(f"Serializing weird numerical {self!r}")
/home/dwillimetz/.local/lib/python3.13/site-packages/rdflib/term.py:1715: UserWarning: Serializing weird numerical rdflib.term.Literal('<0.001', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal'))
  warnings.warn(f"Serializing weird numerical {self!r}")


SystemExit: SHACL validation failed. Inspect the report for details.

/home/dwillimetz/.local/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## SHACL validation results

The materialized RDF graph did not conform to all SHACL constraints. The reported violations identify incomplete values, censored measurements, and redox potentials reported against a reference system outside the currently supported reference list. These findings do not alter the published source data; rather, they identify records that require additional interpretation, contextualisation, or data completion before being used in quantitative cross-study analyses.

| Validation category | Observed issue | Interpretation |
|---|---|---|
| Missing redox-potential values | Several resources typed as `ofbo:OFBO_0001104` do not contain a `qudt:numericValue`. These occur primarily in records from the *Biochimica et Biophysica Acta* dataset (`j_bbabio_2022_148558`), the QM/MM records from `10_1021acs_jpcc_6b07558`, and selected records from other curated studies. | These violations indicate missing numerical data points in the underlying curated tables. They identify entries for which a reported value could potentially be recovered from the original publication or supplementary information. Until a value is added, these records should not be included in analyses requiring a numerical redox potential. |
| Censored solubility values | Several solubility records are reported as upper limits, for example `< 0.01` or `<0.001`, rather than as exact numerical measurements. | These are censored measurements: the source reports that the solubility is below a stated detection or quantification limit, rather than providing an exact concentration. Such values cannot be represented as `xsd:decimal` values using `qudt:numericValue`. The violations therefore indicate a need for an explicit representation of inequality-qualified values or detection limits in a future extension of the data model. |
| Missing diffusion-coefficient values | Several diffusion-coefficient records do not contain a `qudt:numericValue`, including CV and RDE diffusion-coefficient records from `10_1021acsenergylett_1c02504` and one record from `10_1002adfm_202312608`. | As for missing redox-potential values, these violations indicate missing numerical data points in the curated source tables. They identify measurements for which a diffusion coefficient was not available in the structured dataset and which could be completed through further review of the source material where possible. |
| Potential references outside the selected standard-reference list | Seven calculated redox-potential records from `10_1039D1TA03655F` are reported relative to `1,4-DHP`, which is not included in the selected list of standard reference systems. | These records use a study-specific computational reference rather than one of the selected standard reference systems (`SHE`, `NHE`, `Ag/AgCl`, `SCE`, or `Fc/Fc+`). The violation identifies values that require a documented reference-scale conversion before direct numerical comparison with measurements reported on the selected standard scales. The originally reported reference is retained as provenance metadata. 

Overall, the validation demonstrates that SHACL can identify missing quantitative records, values requiring specialised representation, and electrochemical values that are not directly comparable because they use a different reference system. The complete machine-readable validation report is available in `shacl-report.ttl`.